In [32]:
import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.utils import shuffle
from sklearn.model_selection import ParameterGrid

## Load Dataset

In [33]:
train_data = pd.read_csv("training_set_VU_DM.csv")
test_data = pd.read_csv("test_set_VU_DM.csv")

## Feature Engineering

### Missing value estimation

We want feature with monotonic utility with respect to target variable

#### On hotel descriptions
User do not like to book and click hotels with miss values. Filling missing values in hotel description with worse case scenario

In [34]:
data_true = train_data[train_data['booking_bool'] == True]
numeric_col = ["prop_starrating", "prop_review_score", "prop_location_score1", "prop_location_score2", "prop_log_historical_price", "price_usd"]
for col in numeric_col:
    mean_value = data_true[col].mean()
    train_data[col] = train_data[col].fillna(mean_value)
    test_data[col] = test_data[col].fillna(mean_value)
    
categorical_col = ["prop_brand_bool", "prop_brand_bool", "promotion_flag"]
for col in categorical_col:
    mode_val = data_true[col].mode().iloc[0]
    train_data[col] = train_data[col].fillna(mode_val)
    test_data[col] = test_data[col].fillna(mode_val)

#### On user's Historical Data

using hotels data to fill

In [35]:
train_data["visitor_hist_starrating"] = train_data["visitor_hist_starrating"].fillna(0)
train_data["prop_starrating"] = train_data["prop_starrating"].fillna(0)
test_data["visitor_hist_starrating"] = test_data["visitor_hist_starrating"].fillna(0)
test_data["prop_starrating"] = test_data["prop_starrating"].fillna(0)

train_data["starring_diff"] = (train_data["visitor_hist_starrating"] - train_data["prop_starrating"]).abs()
test_data["starring_diff"] = (test_data["visitor_hist_starrating"] - test_data["prop_starrating"]).abs()

train_data["visitor_hist_adr_usd"] = train_data["visitor_hist_adr_usd"].fillna(0)
train_data["price_usd"] = train_data["price_usd"].fillna(0)
test_data["visitor_hist_adr_usd"] = test_data["visitor_hist_adr_usd"].fillna(0)
test_data["price_usd"] = test_data["price_usd"].fillna(0)

train_data["user_diff"] = (train_data["visitor_hist_adr_usd"] - train_data["price_usd"]).abs()
test_data["user_diff"] = (test_data["visitor_hist_adr_usd"] - test_data["price_usd"]).abs()

#### On competitor descriptions
Filling with zero

In [36]:
comp_col_names = ["comp1_rate", "comp1_inv", "comp1_rate_percent_diff", "comp2_rate", "comp2_inv", "comp2_rate_percent_diff", "comp3_rate", "comp3_inv", "comp3_rate_percent_diff", "comp4_rate", "comp4_inv", "comp4_rate_percent_diff", "comp5_rate", "comp5_inv", "comp5_rate_percent_diff", "comp6_rate", "comp6_inv", "comp6_rate_percent_diff", "comp7_rate", "comp7_inv", "comp7_rate_percent_diff", "comp8_rate", "comp8_inv", "comp8_rate_percent_diff"]
for col in comp_col_names:
    train_data[col] = train_data[col].fillna(0)
    test_data[col] = test_data[col].fillna(0)

In [37]:
train_data = train_data.fillna(-1)
test_data = test_data.fillna(-1)

### Feature extraction

In [38]:
grouped = train_data.groupby('prop_id')['booking_bool'].agg(total_count='count', total_booked='sum')
grouped['booking_ratio'] = grouped['total_booked'] / grouped['total_count']
grouped = grouped.reset_index()
train_data = train_data.merge(grouped[['prop_id', 'booking_ratio']], on='prop_id', how='left')
test_data = test_data.merge(grouped[['prop_id', 'booking_ratio']], on='prop_id', how='left')

grouped = train_data.groupby('prop_id')['click_bool'].agg(total_count='count', total_clicked='sum')
grouped['clicking_ratio'] = grouped['total_clicked'] / grouped['total_count']
grouped = grouped.reset_index()
train_data = train_data.merge(grouped[['prop_id', 'clicking_ratio']], on='prop_id', how='left')
test_data = test_data.merge(grouped[['prop_id', 'clicking_ratio']], on='prop_id', how='left')

extract mean, std, median values

In [39]:
columns_to_calculate = [
    'prop_starrating', 'prop_review_score', 'prop_location_score1',
    'prop_location_score2', 'prop_log_historical_price', 'price_usd'
]

data = pd.concat([train_data, test_data], axis=0)

statistics = data.groupby('prop_id')[columns_to_calculate].agg(['mean', 'std', 'median'])
new_column_names = [f"{col}_{stat}" for col in columns_to_calculate for stat in ['mean', 'std', 'median']]
statistics.columns = new_column_names

train_data = train_data.merge(statistics, on='prop_id', how='left')
test_data = test_data.merge(statistics, on='prop_id', how='left')

extract season and weekday

In [40]:
def get_season(month):
    if 3 <= month <= 5:
        return 2
    elif 6 <= month <= 8:
        return 3
    elif 9 <= month <= 11:
        return 1
    else:
        return 0

In [41]:
train_data['date_time'] = pd.to_datetime(train_data['date_time'])
train_data['weekday'] = train_data['date_time'].dt.dayofweek
train_data['season'] = train_data['date_time'].dt.month.apply(get_season)
train_data.drop(['date_time'], axis=1, inplace=True)

test_data['date_time'] = pd.to_datetime(test_data['date_time'])
test_data['weekday'] = test_data['date_time'].dt.dayofweek
test_data['season'] = test_data['date_time'].dt.month.apply(get_season)
test_data.drop(['date_time'], axis=1, inplace=True)

In [42]:
train_data

,srch_id,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,prop_brand_bool,...,prop_location_score2_std,prop_location_score2_median,prop_log_historical_price_mean,prop_log_historical_price_std,prop_log_historical_price_median,price_usd_mean,price_usd_std,price_usd_median,weekday,season
0,1,12,187,0.0,0.0,219,893,3,3.5,1,...,0.040963,0.0438,4.197753,1.794532,4.96,128.188674,274.352024,118.000,3,2
1,1,12,187,0.0,0.0,219,10404,4,4.0,1,...,0.051066,0.0149,4.382724,1.684869,5.03,151.229867,373.115353,129.000,3,2
2,1,12,187,0.0,0.0,219,21315,3,4.5,1,...,0.062955,0.0245,4.288520,1.657737,4.92,171.817203,411.915408,162.000,3,2
3,1,12,187,0.0,0.0,219,27348,2,4.0,1,...,0.030339,0.0125,3.956847,1.322878,4.40,78.347484,220.698689,65.505,3,2
4,1,12,187,0.0,0.0,219,29604,4,3.5,1,...,0.064117,0.1241,4.270591,1.664130,4.91,138.706841,400.190399,118.840,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4958342,332785,5,219,0.0,0.0,219,77700,3,4.0,1,...,0.061233,0.0471,4.077732,1.892576,4.99,132.828454,18.503546,130.000,6,3
4958343,332785,5,219,0.0,0.0,219,88083,3,4.0,1,...,0.041382,0.1520,4.016250,1.528309,4.58,84.609250,13.939205,89.000,6,3
4958344,332785,5,219,0.0,0.0,219,94508,3,3.5,1,...,0.054081,0.0164,4.050111,1.822534,4.87,113.623333,19.685916,109.000,6,3
4958345,332785,5,219,0.0,0.0,219,128360,3,5.0,1,...,0.051853,0.0662,4.773099,1.323157,5.14,149.583944,25.100974,139.000,6,3


### Feature Normalizations

In [12]:
col_to_normalize = comp_col_names + [
    'prop_starrating', 'prop_review_score', 'prop_location_score1',
    'prop_location_score2', 'prop_log_historical_price', 'price_usd'
]

indicator_list = ["srch_id", "prop_id", "season", "srch_booking_window", "srch_destination_id", "prop_country_id"]

In [13]:
def normalize_group(data, group_col, target_col):
    group_mean = data.groupby(group_col)[target_col].transform('mean')
    group_std = data.groupby(group_col)[target_col].transform('std')
    return (data[target_col] - group_mean) / group_std

In [14]:
# for indicator in indicator_list:
#     for col in col_to_normalize:
#         print(f"'{indicator}_{col}': normalize_group(test_data, '{indicator}', '{col}'),")

In [15]:
new_train_cols = pd.DataFrame({
    'srch_id_comp1_rate': normalize_group(train_data, 'srch_id', 'comp1_rate'),
    'srch_id_comp1_inv': normalize_group(train_data, 'srch_id', 'comp1_inv'),
    'srch_id_comp1_rate_percent_diff': normalize_group(train_data, 'srch_id', 'comp1_rate_percent_diff'),
    'srch_id_comp2_rate': normalize_group(train_data, 'srch_id', 'comp2_rate'),
    'srch_id_comp2_inv': normalize_group(train_data, 'srch_id', 'comp2_inv'),
    'srch_id_comp2_rate_percent_diff': normalize_group(train_data, 'srch_id', 'comp2_rate_percent_diff'),
    'srch_id_comp3_rate': normalize_group(train_data, 'srch_id', 'comp3_rate'),
    'srch_id_comp3_inv': normalize_group(train_data, 'srch_id', 'comp3_inv'),
    'srch_id_comp3_rate_percent_diff': normalize_group(train_data, 'srch_id', 'comp3_rate_percent_diff'),
    'srch_id_comp4_rate': normalize_group(train_data, 'srch_id', 'comp4_rate'),
    'srch_id_comp4_inv': normalize_group(train_data, 'srch_id', 'comp4_inv'),
    'srch_id_comp4_rate_percent_diff': normalize_group(train_data, 'srch_id', 'comp4_rate_percent_diff'),
    'srch_id_comp5_rate': normalize_group(train_data, 'srch_id', 'comp5_rate'),
    'srch_id_comp5_inv': normalize_group(train_data, 'srch_id', 'comp5_inv'),
    'srch_id_comp5_rate_percent_diff': normalize_group(train_data, 'srch_id', 'comp5_rate_percent_diff'),
    'srch_id_comp6_rate': normalize_group(train_data, 'srch_id', 'comp6_rate'),
    'srch_id_comp6_inv': normalize_group(train_data, 'srch_id', 'comp6_inv'),
    'srch_id_comp6_rate_percent_diff': normalize_group(train_data, 'srch_id', 'comp6_rate_percent_diff'),
    'srch_id_comp7_rate': normalize_group(train_data, 'srch_id', 'comp7_rate'),
    'srch_id_comp7_inv': normalize_group(train_data, 'srch_id', 'comp7_inv'),
    'srch_id_comp7_rate_percent_diff': normalize_group(train_data, 'srch_id', 'comp7_rate_percent_diff'),
    'srch_id_comp8_rate': normalize_group(train_data, 'srch_id', 'comp8_rate'),
    'srch_id_comp8_inv': normalize_group(train_data, 'srch_id', 'comp8_inv'),
    'srch_id_comp8_rate_percent_diff': normalize_group(train_data, 'srch_id', 'comp8_rate_percent_diff'),
    'srch_id_prop_starrating': normalize_group(train_data, 'srch_id', 'prop_starrating'),
    'srch_id_prop_review_score': normalize_group(train_data, 'srch_id', 'prop_review_score'),
    'srch_id_prop_location_score1': normalize_group(train_data, 'srch_id', 'prop_location_score1'),
    'srch_id_prop_location_score2': normalize_group(train_data, 'srch_id', 'prop_location_score2'),
    'srch_id_prop_log_historical_price': normalize_group(train_data, 'srch_id', 'prop_log_historical_price'),
    'srch_id_price_usd': normalize_group(train_data, 'srch_id', 'price_usd'),
    'prop_id_comp1_rate': normalize_group(train_data, 'prop_id', 'comp1_rate'),
    'prop_id_comp1_inv': normalize_group(train_data, 'prop_id', 'comp1_inv'),
    'prop_id_comp1_rate_percent_diff': normalize_group(train_data, 'prop_id', 'comp1_rate_percent_diff'),
    'prop_id_comp2_rate': normalize_group(train_data, 'prop_id', 'comp2_rate'),
    'prop_id_comp2_inv': normalize_group(train_data, 'prop_id', 'comp2_inv'),
    'prop_id_comp2_rate_percent_diff': normalize_group(train_data, 'prop_id', 'comp2_rate_percent_diff'),
    'prop_id_comp3_rate': normalize_group(train_data, 'prop_id', 'comp3_rate'),
    'prop_id_comp3_inv': normalize_group(train_data, 'prop_id', 'comp3_inv'),
    'prop_id_comp3_rate_percent_diff': normalize_group(train_data, 'prop_id', 'comp3_rate_percent_diff'),
    'prop_id_comp4_rate': normalize_group(train_data, 'prop_id', 'comp4_rate'),
    'prop_id_comp4_inv': normalize_group(train_data, 'prop_id', 'comp4_inv'),
    'prop_id_comp4_rate_percent_diff': normalize_group(train_data, 'prop_id', 'comp4_rate_percent_diff'),
    'prop_id_comp5_rate': normalize_group(train_data, 'prop_id', 'comp5_rate'),
    'prop_id_comp5_inv': normalize_group(train_data, 'prop_id', 'comp5_inv'),
    'prop_id_comp5_rate_percent_diff': normalize_group(train_data, 'prop_id', 'comp5_rate_percent_diff'),
    'prop_id_comp6_rate': normalize_group(train_data, 'prop_id', 'comp6_rate'),
    'prop_id_comp6_inv': normalize_group(train_data, 'prop_id', 'comp6_inv'),
    'prop_id_comp6_rate_percent_diff': normalize_group(train_data, 'prop_id', 'comp6_rate_percent_diff'),
    'prop_id_comp7_rate': normalize_group(train_data, 'prop_id', 'comp7_rate'),
    'prop_id_comp7_inv': normalize_group(train_data, 'prop_id', 'comp7_inv'),
    'prop_id_comp7_rate_percent_diff': normalize_group(train_data, 'prop_id', 'comp7_rate_percent_diff'),
    'prop_id_comp8_rate': normalize_group(train_data, 'prop_id', 'comp8_rate'),
    'prop_id_comp8_inv': normalize_group(train_data, 'prop_id', 'comp8_inv'),
    'prop_id_comp8_rate_percent_diff': normalize_group(train_data, 'prop_id', 'comp8_rate_percent_diff'),
    'prop_id_prop_starrating': normalize_group(train_data, 'prop_id', 'prop_starrating'),
    'prop_id_prop_review_score': normalize_group(train_data, 'prop_id', 'prop_review_score'),
    'prop_id_prop_location_score1': normalize_group(train_data, 'prop_id', 'prop_location_score1'),
    'prop_id_prop_location_score2': normalize_group(train_data, 'prop_id', 'prop_location_score2'),
    'prop_id_prop_log_historical_price': normalize_group(train_data, 'prop_id', 'prop_log_historical_price'),
    'prop_id_price_usd': normalize_group(train_data, 'prop_id', 'price_usd'),
    'season_comp1_rate': normalize_group(train_data, 'season', 'comp1_rate'),
    'season_comp1_inv': normalize_group(train_data, 'season', 'comp1_inv'),
    'season_comp1_rate_percent_diff': normalize_group(train_data, 'season', 'comp1_rate_percent_diff'),
    'season_comp2_rate': normalize_group(train_data, 'season', 'comp2_rate'),
    'season_comp2_inv': normalize_group(train_data, 'season', 'comp2_inv'),
    'season_comp2_rate_percent_diff': normalize_group(train_data, 'season', 'comp2_rate_percent_diff'),
    'season_comp3_rate': normalize_group(train_data, 'season', 'comp3_rate'),
    'season_comp3_inv': normalize_group(train_data, 'season', 'comp3_inv'),
    'season_comp3_rate_percent_diff': normalize_group(train_data, 'season', 'comp3_rate_percent_diff'),
    'season_comp4_rate': normalize_group(train_data, 'season', 'comp4_rate'),
    'season_comp4_inv': normalize_group(train_data, 'season', 'comp4_inv'),
    'season_comp4_rate_percent_diff': normalize_group(train_data, 'season', 'comp4_rate_percent_diff'),
    'season_comp5_rate': normalize_group(train_data, 'season', 'comp5_rate'),
    'season_comp5_inv': normalize_group(train_data, 'season', 'comp5_inv'),
    'season_comp5_rate_percent_diff': normalize_group(train_data, 'season', 'comp5_rate_percent_diff'),
    'season_comp6_rate': normalize_group(train_data, 'season', 'comp6_rate'),
    'season_comp6_inv': normalize_group(train_data, 'season', 'comp6_inv'),
    'season_comp6_rate_percent_diff': normalize_group(train_data, 'season', 'comp6_rate_percent_diff'),
    'season_comp7_rate': normalize_group(train_data, 'season', 'comp7_rate'),
    'season_comp7_inv': normalize_group(train_data, 'season', 'comp7_inv'),
    'season_comp7_rate_percent_diff': normalize_group(train_data, 'season', 'comp7_rate_percent_diff'),
    'season_comp8_rate': normalize_group(train_data, 'season', 'comp8_rate'),
    'season_comp8_inv': normalize_group(train_data, 'season', 'comp8_inv'),
    'season_comp8_rate_percent_diff': normalize_group(train_data, 'season', 'comp8_rate_percent_diff'),
    'season_prop_starrating': normalize_group(train_data, 'season', 'prop_starrating'),
    'season_prop_review_score': normalize_group(train_data, 'season', 'prop_review_score'),
    'season_prop_location_score1': normalize_group(train_data, 'season', 'prop_location_score1'),
    'season_prop_location_score2': normalize_group(train_data, 'season', 'prop_location_score2'),
    'season_prop_log_historical_price': normalize_group(train_data, 'season', 'prop_log_historical_price'),
    'season_price_usd': normalize_group(train_data, 'season', 'price_usd'),
    'srch_booking_window_comp1_rate': normalize_group(train_data, 'srch_booking_window', 'comp1_rate'),
    'srch_booking_window_comp1_inv': normalize_group(train_data, 'srch_booking_window', 'comp1_inv'),
    'srch_booking_window_comp1_rate_percent_diff': normalize_group(train_data, 'srch_booking_window', 'comp1_rate_percent_diff'),
    'srch_booking_window_comp2_rate': normalize_group(train_data, 'srch_booking_window', 'comp2_rate'),
    'srch_booking_window_comp2_inv': normalize_group(train_data, 'srch_booking_window', 'comp2_inv'),
    'srch_booking_window_comp2_rate_percent_diff': normalize_group(train_data, 'srch_booking_window', 'comp2_rate_percent_diff'),
    'srch_booking_window_comp3_rate': normalize_group(train_data, 'srch_booking_window', 'comp3_rate'),
    'srch_booking_window_comp3_inv': normalize_group(train_data, 'srch_booking_window', 'comp3_inv'),
    'srch_booking_window_comp3_rate_percent_diff': normalize_group(train_data, 'srch_booking_window', 'comp3_rate_percent_diff'),
    'srch_booking_window_comp4_rate': normalize_group(train_data, 'srch_booking_window', 'comp4_rate'),
    'srch_booking_window_comp4_inv': normalize_group(train_data, 'srch_booking_window', 'comp4_inv'),
    'srch_booking_window_comp4_rate_percent_diff': normalize_group(train_data, 'srch_booking_window', 'comp4_rate_percent_diff'),
    'srch_booking_window_comp5_rate': normalize_group(train_data, 'srch_booking_window', 'comp5_rate'),
    'srch_booking_window_comp5_inv': normalize_group(train_data, 'srch_booking_window', 'comp5_inv'),
    'srch_booking_window_comp5_rate_percent_diff': normalize_group(train_data, 'srch_booking_window', 'comp5_rate_percent_diff'),
    'srch_booking_window_comp6_rate': normalize_group(train_data, 'srch_booking_window', 'comp6_rate'),
    'srch_booking_window_comp6_inv': normalize_group(train_data, 'srch_booking_window', 'comp6_inv'),
    'srch_booking_window_comp6_rate_percent_diff': normalize_group(train_data, 'srch_booking_window', 'comp6_rate_percent_diff'),
    'srch_booking_window_comp7_rate': normalize_group(train_data, 'srch_booking_window', 'comp7_rate'),
    'srch_booking_window_comp7_inv': normalize_group(train_data, 'srch_booking_window', 'comp7_inv'),
    'srch_booking_window_comp7_rate_percent_diff': normalize_group(train_data, 'srch_booking_window', 'comp7_rate_percent_diff'),
    'srch_booking_window_comp8_rate': normalize_group(train_data, 'srch_booking_window', 'comp8_rate'),
    'srch_booking_window_comp8_inv': normalize_group(train_data, 'srch_booking_window', 'comp8_inv'),
    'srch_booking_window_comp8_rate_percent_diff': normalize_group(train_data, 'srch_booking_window', 'comp8_rate_percent_diff'),
    'srch_booking_window_prop_starrating': normalize_group(train_data, 'srch_booking_window', 'prop_starrating'),
    'srch_booking_window_prop_review_score': normalize_group(train_data, 'srch_booking_window', 'prop_review_score'),
    'srch_booking_window_prop_location_score1': normalize_group(train_data, 'srch_booking_window', 'prop_location_score1'),
    'srch_booking_window_prop_location_score2': normalize_group(train_data, 'srch_booking_window', 'prop_location_score2'),
    'srch_booking_window_prop_log_historical_price': normalize_group(train_data, 'srch_booking_window', 'prop_log_historical_price'),
    'srch_booking_window_price_usd': normalize_group(train_data, 'srch_booking_window', 'price_usd'),
    'srch_destination_id_comp1_rate': normalize_group(train_data, 'srch_destination_id', 'comp1_rate'),
    'srch_destination_id_comp1_inv': normalize_group(train_data, 'srch_destination_id', 'comp1_inv'),
    'srch_destination_id_comp1_rate_percent_diff': normalize_group(train_data, 'srch_destination_id', 'comp1_rate_percent_diff'),
    'srch_destination_id_comp2_rate': normalize_group(train_data, 'srch_destination_id', 'comp2_rate'),
    'srch_destination_id_comp2_inv': normalize_group(train_data, 'srch_destination_id', 'comp2_inv'),
    'srch_destination_id_comp2_rate_percent_diff': normalize_group(train_data, 'srch_destination_id', 'comp2_rate_percent_diff'),
    'srch_destination_id_comp3_rate': normalize_group(train_data, 'srch_destination_id', 'comp3_rate'),
    'srch_destination_id_comp3_inv': normalize_group(train_data, 'srch_destination_id', 'comp3_inv'),
    'srch_destination_id_comp3_rate_percent_diff': normalize_group(train_data, 'srch_destination_id', 'comp3_rate_percent_diff'),
    'srch_destination_id_comp4_rate': normalize_group(train_data, 'srch_destination_id', 'comp4_rate'),
    'srch_destination_id_comp4_inv': normalize_group(train_data, 'srch_destination_id', 'comp4_inv'),
    'srch_destination_id_comp4_rate_percent_diff': normalize_group(train_data, 'srch_destination_id', 'comp4_rate_percent_diff'),
    'srch_destination_id_comp5_rate': normalize_group(train_data, 'srch_destination_id', 'comp5_rate'),
    'srch_destination_id_comp5_inv': normalize_group(train_data, 'srch_destination_id', 'comp5_inv'),
    'srch_destination_id_comp5_rate_percent_diff': normalize_group(train_data, 'srch_destination_id', 'comp5_rate_percent_diff'),
    'srch_destination_id_comp6_rate': normalize_group(train_data, 'srch_destination_id', 'comp6_rate'),
    'srch_destination_id_comp6_inv': normalize_group(train_data, 'srch_destination_id', 'comp6_inv'),
    'srch_destination_id_comp6_rate_percent_diff': normalize_group(train_data, 'srch_destination_id', 'comp6_rate_percent_diff'),
    'srch_destination_id_comp7_rate': normalize_group(train_data, 'srch_destination_id', 'comp7_rate'),
    'srch_destination_id_comp7_inv': normalize_group(train_data, 'srch_destination_id', 'comp7_inv'),
    'srch_destination_id_comp7_rate_percent_diff': normalize_group(train_data, 'srch_destination_id', 'comp7_rate_percent_diff'),
    'srch_destination_id_comp8_rate': normalize_group(train_data, 'srch_destination_id', 'comp8_rate'),
    'srch_destination_id_comp8_inv': normalize_group(train_data, 'srch_destination_id', 'comp8_inv'),
    'srch_destination_id_comp8_rate_percent_diff': normalize_group(train_data, 'srch_destination_id', 'comp8_rate_percent_diff'),
    'srch_destination_id_prop_starrating': normalize_group(train_data, 'srch_destination_id', 'prop_starrating'),
    'srch_destination_id_prop_review_score': normalize_group(train_data, 'srch_destination_id', 'prop_review_score'),
    'srch_destination_id_prop_location_score1': normalize_group(train_data, 'srch_destination_id', 'prop_location_score1'),
    'srch_destination_id_prop_location_score2': normalize_group(train_data, 'srch_destination_id', 'prop_location_score2'),
    'srch_destination_id_prop_log_historical_price': normalize_group(train_data, 'srch_destination_id', 'prop_log_historical_price'),
    'srch_destination_id_price_usd': normalize_group(train_data, 'srch_destination_id', 'price_usd'),
    'prop_country_id_comp1_rate': normalize_group(train_data, 'prop_country_id', 'comp1_rate'),
    'prop_country_id_comp1_inv': normalize_group(train_data, 'prop_country_id', 'comp1_inv'),
    'prop_country_id_comp1_rate_percent_diff': normalize_group(train_data, 'prop_country_id', 'comp1_rate_percent_diff'),
    'prop_country_id_comp2_rate': normalize_group(train_data, 'prop_country_id', 'comp2_rate'),
    'prop_country_id_comp2_inv': normalize_group(train_data, 'prop_country_id', 'comp2_inv'),
    'prop_country_id_comp2_rate_percent_diff': normalize_group(train_data, 'prop_country_id', 'comp2_rate_percent_diff'),
    'prop_country_id_comp3_rate': normalize_group(train_data, 'prop_country_id', 'comp3_rate'),
    'prop_country_id_comp3_inv': normalize_group(train_data, 'prop_country_id', 'comp3_inv'),
    'prop_country_id_comp3_rate_percent_diff': normalize_group(train_data, 'prop_country_id', 'comp3_rate_percent_diff'),
    'prop_country_id_comp4_rate': normalize_group(train_data, 'prop_country_id', 'comp4_rate'),
    'prop_country_id_comp4_inv': normalize_group(train_data, 'prop_country_id', 'comp4_inv'),
    'prop_country_id_comp4_rate_percent_diff': normalize_group(train_data, 'prop_country_id', 'comp4_rate_percent_diff'),
    'prop_country_id_comp5_rate': normalize_group(train_data, 'prop_country_id', 'comp5_rate'),
    'prop_country_id_comp5_inv': normalize_group(train_data, 'prop_country_id', 'comp5_inv'),
    'prop_country_id_comp5_rate_percent_diff': normalize_group(train_data, 'prop_country_id', 'comp5_rate_percent_diff'),
    'prop_country_id_comp6_rate': normalize_group(train_data, 'prop_country_id', 'comp6_rate'),
    'prop_country_id_comp6_inv': normalize_group(train_data, 'prop_country_id', 'comp6_inv'),
    'prop_country_id_comp6_rate_percent_diff': normalize_group(train_data, 'prop_country_id', 'comp6_rate_percent_diff'),
    'prop_country_id_comp7_rate': normalize_group(train_data, 'prop_country_id', 'comp7_rate'),
    'prop_country_id_comp7_inv': normalize_group(train_data, 'prop_country_id', 'comp7_inv'),
    'prop_country_id_comp7_rate_percent_diff': normalize_group(train_data, 'prop_country_id', 'comp7_rate_percent_diff'),
    'prop_country_id_comp8_rate': normalize_group(train_data, 'prop_country_id', 'comp8_rate'),
    'prop_country_id_comp8_inv': normalize_group(train_data, 'prop_country_id', 'comp8_inv'),
    'prop_country_id_comp8_rate_percent_diff': normalize_group(train_data, 'prop_country_id', 'comp8_rate_percent_diff'),
    'prop_country_id_prop_starrating': normalize_group(train_data, 'prop_country_id', 'prop_starrating'),
    'prop_country_id_prop_review_score': normalize_group(train_data, 'prop_country_id', 'prop_review_score'),
    'prop_country_id_prop_location_score1': normalize_group(train_data, 'prop_country_id', 'prop_location_score1'),
    'prop_country_id_prop_location_score2': normalize_group(train_data, 'prop_country_id', 'prop_location_score2'),
    'prop_country_id_prop_log_historical_price': normalize_group(train_data, 'prop_country_id', 'prop_log_historical_price'),
    'prop_country_id_price_usd': normalize_group(train_data, 'prop_country_id', 'price_usd'),
})

In [17]:
new_test_cols = pd.DataFrame({
    'srch_id_comp1_rate': normalize_group(test_data, 'srch_id', 'comp1_rate'),
'srch_id_comp1_inv': normalize_group(test_data, 'srch_id', 'comp1_inv'),
'srch_id_comp1_rate_percent_diff': normalize_group(test_data, 'srch_id', 'comp1_rate_percent_diff'),
'srch_id_comp2_rate': normalize_group(test_data, 'srch_id', 'comp2_rate'),
'srch_id_comp2_inv': normalize_group(test_data, 'srch_id', 'comp2_inv'),
'srch_id_comp2_rate_percent_diff': normalize_group(test_data, 'srch_id', 'comp2_rate_percent_diff'),
'srch_id_comp3_rate': normalize_group(test_data, 'srch_id', 'comp3_rate'),
'srch_id_comp3_inv': normalize_group(test_data, 'srch_id', 'comp3_inv'),
'srch_id_comp3_rate_percent_diff': normalize_group(test_data, 'srch_id', 'comp3_rate_percent_diff'),
'srch_id_comp4_rate': normalize_group(test_data, 'srch_id', 'comp4_rate'),
'srch_id_comp4_inv': normalize_group(test_data, 'srch_id', 'comp4_inv'),
'srch_id_comp4_rate_percent_diff': normalize_group(test_data, 'srch_id', 'comp4_rate_percent_diff'),
'srch_id_comp5_rate': normalize_group(test_data, 'srch_id', 'comp5_rate'),
'srch_id_comp5_inv': normalize_group(test_data, 'srch_id', 'comp5_inv'),
'srch_id_comp5_rate_percent_diff': normalize_group(test_data, 'srch_id', 'comp5_rate_percent_diff'),
'srch_id_comp6_rate': normalize_group(test_data, 'srch_id', 'comp6_rate'),
'srch_id_comp6_inv': normalize_group(test_data, 'srch_id', 'comp6_inv'),
'srch_id_comp6_rate_percent_diff': normalize_group(test_data, 'srch_id', 'comp6_rate_percent_diff'),
'srch_id_comp7_rate': normalize_group(test_data, 'srch_id', 'comp7_rate'),
'srch_id_comp7_inv': normalize_group(test_data, 'srch_id', 'comp7_inv'),
'srch_id_comp7_rate_percent_diff': normalize_group(test_data, 'srch_id', 'comp7_rate_percent_diff'),
'srch_id_comp8_rate': normalize_group(test_data, 'srch_id', 'comp8_rate'),
'srch_id_comp8_inv': normalize_group(test_data, 'srch_id', 'comp8_inv'),
'srch_id_comp8_rate_percent_diff': normalize_group(test_data, 'srch_id', 'comp8_rate_percent_diff'),
'srch_id_prop_starrating': normalize_group(test_data, 'srch_id', 'prop_starrating'),
'srch_id_prop_review_score': normalize_group(test_data, 'srch_id', 'prop_review_score'),
'srch_id_prop_location_score1': normalize_group(test_data, 'srch_id', 'prop_location_score1'),
'srch_id_prop_location_score2': normalize_group(test_data, 'srch_id', 'prop_location_score2'),
'srch_id_prop_log_historical_price': normalize_group(test_data, 'srch_id', 'prop_log_historical_price'),
'srch_id_price_usd': normalize_group(test_data, 'srch_id', 'price_usd'),
'prop_id_comp1_rate': normalize_group(test_data, 'prop_id', 'comp1_rate'),
'prop_id_comp1_inv': normalize_group(test_data, 'prop_id', 'comp1_inv'),
'prop_id_comp1_rate_percent_diff': normalize_group(test_data, 'prop_id', 'comp1_rate_percent_diff'),
'prop_id_comp2_rate': normalize_group(test_data, 'prop_id', 'comp2_rate'),
'prop_id_comp2_inv': normalize_group(test_data, 'prop_id', 'comp2_inv'),
'prop_id_comp2_rate_percent_diff': normalize_group(test_data, 'prop_id', 'comp2_rate_percent_diff'),
'prop_id_comp3_rate': normalize_group(test_data, 'prop_id', 'comp3_rate'),
'prop_id_comp3_inv': normalize_group(test_data, 'prop_id', 'comp3_inv'),
'prop_id_comp3_rate_percent_diff': normalize_group(test_data, 'prop_id', 'comp3_rate_percent_diff'),
'prop_id_comp4_rate': normalize_group(test_data, 'prop_id', 'comp4_rate'),
'prop_id_comp4_inv': normalize_group(test_data, 'prop_id', 'comp4_inv'),
'prop_id_comp4_rate_percent_diff': normalize_group(test_data, 'prop_id', 'comp4_rate_percent_diff'),
'prop_id_comp5_rate': normalize_group(test_data, 'prop_id', 'comp5_rate'),
'prop_id_comp5_inv': normalize_group(test_data, 'prop_id', 'comp5_inv'),
'prop_id_comp5_rate_percent_diff': normalize_group(test_data, 'prop_id', 'comp5_rate_percent_diff'),
'prop_id_comp6_rate': normalize_group(test_data, 'prop_id', 'comp6_rate'),
'prop_id_comp6_inv': normalize_group(test_data, 'prop_id', 'comp6_inv'),
'prop_id_comp6_rate_percent_diff': normalize_group(test_data, 'prop_id', 'comp6_rate_percent_diff'),
'prop_id_comp7_rate': normalize_group(test_data, 'prop_id', 'comp7_rate'),
'prop_id_comp7_inv': normalize_group(test_data, 'prop_id', 'comp7_inv'),
'prop_id_comp7_rate_percent_diff': normalize_group(test_data, 'prop_id', 'comp7_rate_percent_diff'),
'prop_id_comp8_rate': normalize_group(test_data, 'prop_id', 'comp8_rate'),
'prop_id_comp8_inv': normalize_group(test_data, 'prop_id', 'comp8_inv'),
'prop_id_comp8_rate_percent_diff': normalize_group(test_data, 'prop_id', 'comp8_rate_percent_diff'),
'prop_id_prop_starrating': normalize_group(test_data, 'prop_id', 'prop_starrating'),
'prop_id_prop_review_score': normalize_group(test_data, 'prop_id', 'prop_review_score'),
'prop_id_prop_location_score1': normalize_group(test_data, 'prop_id', 'prop_location_score1'),
'prop_id_prop_location_score2': normalize_group(test_data, 'prop_id', 'prop_location_score2'),
'prop_id_prop_log_historical_price': normalize_group(test_data, 'prop_id', 'prop_log_historical_price'),
'prop_id_price_usd': normalize_group(test_data, 'prop_id', 'price_usd'),
'season_comp1_rate': normalize_group(test_data, 'season', 'comp1_rate'),
'season_comp1_inv': normalize_group(test_data, 'season', 'comp1_inv'),
'season_comp1_rate_percent_diff': normalize_group(test_data, 'season', 'comp1_rate_percent_diff'),
'season_comp2_rate': normalize_group(test_data, 'season', 'comp2_rate'),
'season_comp2_inv': normalize_group(test_data, 'season', 'comp2_inv'),
'season_comp2_rate_percent_diff': normalize_group(test_data, 'season', 'comp2_rate_percent_diff'),
'season_comp3_rate': normalize_group(test_data, 'season', 'comp3_rate'),
'season_comp3_inv': normalize_group(test_data, 'season', 'comp3_inv'),
'season_comp3_rate_percent_diff': normalize_group(test_data, 'season', 'comp3_rate_percent_diff'),
'season_comp4_rate': normalize_group(test_data, 'season', 'comp4_rate'),
'season_comp4_inv': normalize_group(test_data, 'season', 'comp4_inv'),
'season_comp4_rate_percent_diff': normalize_group(test_data, 'season', 'comp4_rate_percent_diff'),
'season_comp5_rate': normalize_group(test_data, 'season', 'comp5_rate'),
'season_comp5_inv': normalize_group(test_data, 'season', 'comp5_inv'),
'season_comp5_rate_percent_diff': normalize_group(test_data, 'season', 'comp5_rate_percent_diff'),
'season_comp6_rate': normalize_group(test_data, 'season', 'comp6_rate'),
'season_comp6_inv': normalize_group(test_data, 'season', 'comp6_inv'),
'season_comp6_rate_percent_diff': normalize_group(test_data, 'season', 'comp6_rate_percent_diff'),
'season_comp7_rate': normalize_group(test_data, 'season', 'comp7_rate'),
'season_comp7_inv': normalize_group(test_data, 'season', 'comp7_inv'),
'season_comp7_rate_percent_diff': normalize_group(test_data, 'season', 'comp7_rate_percent_diff'),
'season_comp8_rate': normalize_group(test_data, 'season', 'comp8_rate'),
'season_comp8_inv': normalize_group(test_data, 'season', 'comp8_inv'),
'season_comp8_rate_percent_diff': normalize_group(test_data, 'season', 'comp8_rate_percent_diff'),
'season_prop_starrating': normalize_group(test_data, 'season', 'prop_starrating'),
'season_prop_review_score': normalize_group(test_data, 'season', 'prop_review_score'),
'season_prop_location_score1': normalize_group(test_data, 'season', 'prop_location_score1'),
'season_prop_location_score2': normalize_group(test_data, 'season', 'prop_location_score2'),
'season_prop_log_historical_price': normalize_group(test_data, 'season', 'prop_log_historical_price'),
'season_price_usd': normalize_group(test_data, 'season', 'price_usd'),
'srch_booking_window_comp1_rate': normalize_group(test_data, 'srch_booking_window', 'comp1_rate'),
'srch_booking_window_comp1_inv': normalize_group(test_data, 'srch_booking_window', 'comp1_inv'),
'srch_booking_window_comp1_rate_percent_diff': normalize_group(test_data, 'srch_booking_window', 'comp1_rate_percent_diff'),
'srch_booking_window_comp2_rate': normalize_group(test_data, 'srch_booking_window', 'comp2_rate'),
'srch_booking_window_comp2_inv': normalize_group(test_data, 'srch_booking_window', 'comp2_inv'),
'srch_booking_window_comp2_rate_percent_diff': normalize_group(test_data, 'srch_booking_window', 'comp2_rate_percent_diff'),
'srch_booking_window_comp3_rate': normalize_group(test_data, 'srch_booking_window', 'comp3_rate'),
'srch_booking_window_comp3_inv': normalize_group(test_data, 'srch_booking_window', 'comp3_inv'),
'srch_booking_window_comp3_rate_percent_diff': normalize_group(test_data, 'srch_booking_window', 'comp3_rate_percent_diff'),
'srch_booking_window_comp4_rate': normalize_group(test_data, 'srch_booking_window', 'comp4_rate'),
'srch_booking_window_comp4_inv': normalize_group(test_data, 'srch_booking_window', 'comp4_inv'),
'srch_booking_window_comp4_rate_percent_diff': normalize_group(test_data, 'srch_booking_window', 'comp4_rate_percent_diff'),
'srch_booking_window_comp5_rate': normalize_group(test_data, 'srch_booking_window', 'comp5_rate'),
'srch_booking_window_comp5_inv': normalize_group(test_data, 'srch_booking_window', 'comp5_inv'),
'srch_booking_window_comp5_rate_percent_diff': normalize_group(test_data, 'srch_booking_window', 'comp5_rate_percent_diff'),
'srch_booking_window_comp6_rate': normalize_group(test_data, 'srch_booking_window', 'comp6_rate'),
'srch_booking_window_comp6_inv': normalize_group(test_data, 'srch_booking_window', 'comp6_inv'),
'srch_booking_window_comp6_rate_percent_diff': normalize_group(test_data, 'srch_booking_window', 'comp6_rate_percent_diff'),
'srch_booking_window_comp7_rate': normalize_group(test_data, 'srch_booking_window', 'comp7_rate'),
'srch_booking_window_comp7_inv': normalize_group(test_data, 'srch_booking_window', 'comp7_inv'),
'srch_booking_window_comp7_rate_percent_diff': normalize_group(test_data, 'srch_booking_window', 'comp7_rate_percent_diff'),
'srch_booking_window_comp8_rate': normalize_group(test_data, 'srch_booking_window', 'comp8_rate'),
'srch_booking_window_comp8_inv': normalize_group(test_data, 'srch_booking_window', 'comp8_inv'),
'srch_booking_window_comp8_rate_percent_diff': normalize_group(test_data, 'srch_booking_window', 'comp8_rate_percent_diff'),
'srch_booking_window_prop_starrating': normalize_group(test_data, 'srch_booking_window', 'prop_starrating'),
'srch_booking_window_prop_review_score': normalize_group(test_data, 'srch_booking_window', 'prop_review_score'),
'srch_booking_window_prop_location_score1': normalize_group(test_data, 'srch_booking_window', 'prop_location_score1'),
'srch_booking_window_prop_location_score2': normalize_group(test_data, 'srch_booking_window', 'prop_location_score2'),
'srch_booking_window_prop_log_historical_price': normalize_group(test_data, 'srch_booking_window', 'prop_log_historical_price'),
'srch_booking_window_price_usd': normalize_group(test_data, 'srch_booking_window', 'price_usd'),
'srch_destination_id_comp1_rate': normalize_group(test_data, 'srch_destination_id', 'comp1_rate'),
'srch_destination_id_comp1_inv': normalize_group(test_data, 'srch_destination_id', 'comp1_inv'),
'srch_destination_id_comp1_rate_percent_diff': normalize_group(test_data, 'srch_destination_id', 'comp1_rate_percent_diff'),
'srch_destination_id_comp2_rate': normalize_group(test_data, 'srch_destination_id', 'comp2_rate'),
'srch_destination_id_comp2_inv': normalize_group(test_data, 'srch_destination_id', 'comp2_inv'),
'srch_destination_id_comp2_rate_percent_diff': normalize_group(test_data, 'srch_destination_id', 'comp2_rate_percent_diff'),
'srch_destination_id_comp3_rate': normalize_group(test_data, 'srch_destination_id', 'comp3_rate'),
'srch_destination_id_comp3_inv': normalize_group(test_data, 'srch_destination_id', 'comp3_inv'),
'srch_destination_id_comp3_rate_percent_diff': normalize_group(test_data, 'srch_destination_id', 'comp3_rate_percent_diff'),
'srch_destination_id_comp4_rate': normalize_group(test_data, 'srch_destination_id', 'comp4_rate'),
'srch_destination_id_comp4_inv': normalize_group(test_data, 'srch_destination_id', 'comp4_inv'),
'srch_destination_id_comp4_rate_percent_diff': normalize_group(test_data, 'srch_destination_id', 'comp4_rate_percent_diff'),
'srch_destination_id_comp5_rate': normalize_group(test_data, 'srch_destination_id', 'comp5_rate'),
'srch_destination_id_comp5_inv': normalize_group(test_data, 'srch_destination_id', 'comp5_inv'),
'srch_destination_id_comp5_rate_percent_diff': normalize_group(test_data, 'srch_destination_id', 'comp5_rate_percent_diff'),
'srch_destination_id_comp6_rate': normalize_group(test_data, 'srch_destination_id', 'comp6_rate'),
'srch_destination_id_comp6_inv': normalize_group(test_data, 'srch_destination_id', 'comp6_inv'),
'srch_destination_id_comp6_rate_percent_diff': normalize_group(test_data, 'srch_destination_id', 'comp6_rate_percent_diff'),
'srch_destination_id_comp7_rate': normalize_group(test_data, 'srch_destination_id', 'comp7_rate'),
'srch_destination_id_comp7_inv': normalize_group(test_data, 'srch_destination_id', 'comp7_inv'),
'srch_destination_id_comp7_rate_percent_diff': normalize_group(test_data, 'srch_destination_id', 'comp7_rate_percent_diff'),
'srch_destination_id_comp8_rate': normalize_group(test_data, 'srch_destination_id', 'comp8_rate'),
'srch_destination_id_comp8_inv': normalize_group(test_data, 'srch_destination_id', 'comp8_inv'),
'srch_destination_id_comp8_rate_percent_diff': normalize_group(test_data, 'srch_destination_id', 'comp8_rate_percent_diff'),
'srch_destination_id_prop_starrating': normalize_group(test_data, 'srch_destination_id', 'prop_starrating'),
'srch_destination_id_prop_review_score': normalize_group(test_data, 'srch_destination_id', 'prop_review_score'),
'srch_destination_id_prop_location_score1': normalize_group(test_data, 'srch_destination_id', 'prop_location_score1'),
'srch_destination_id_prop_location_score2': normalize_group(test_data, 'srch_destination_id', 'prop_location_score2'),
'srch_destination_id_prop_log_historical_price': normalize_group(test_data, 'srch_destination_id', 'prop_log_historical_price'),
'srch_destination_id_price_usd': normalize_group(test_data, 'srch_destination_id', 'price_usd'),
'prop_country_id_comp1_rate': normalize_group(test_data, 'prop_country_id', 'comp1_rate'),
'prop_country_id_comp1_inv': normalize_group(test_data, 'prop_country_id', 'comp1_inv'),
'prop_country_id_comp1_rate_percent_diff': normalize_group(test_data, 'prop_country_id', 'comp1_rate_percent_diff'),
'prop_country_id_comp2_rate': normalize_group(test_data, 'prop_country_id', 'comp2_rate'),
'prop_country_id_comp2_inv': normalize_group(test_data, 'prop_country_id', 'comp2_inv'),
'prop_country_id_comp2_rate_percent_diff': normalize_group(test_data, 'prop_country_id', 'comp2_rate_percent_diff'),
'prop_country_id_comp3_rate': normalize_group(test_data, 'prop_country_id', 'comp3_rate'),
'prop_country_id_comp3_inv': normalize_group(test_data, 'prop_country_id', 'comp3_inv'),
'prop_country_id_comp3_rate_percent_diff': normalize_group(test_data, 'prop_country_id', 'comp3_rate_percent_diff'),
'prop_country_id_comp4_rate': normalize_group(test_data, 'prop_country_id', 'comp4_rate'),
'prop_country_id_comp4_inv': normalize_group(test_data, 'prop_country_id', 'comp4_inv'),
'prop_country_id_comp4_rate_percent_diff': normalize_group(test_data, 'prop_country_id', 'comp4_rate_percent_diff'),
'prop_country_id_comp5_rate': normalize_group(test_data, 'prop_country_id', 'comp5_rate'),
'prop_country_id_comp5_inv': normalize_group(test_data, 'prop_country_id', 'comp5_inv'),
'prop_country_id_comp5_rate_percent_diff': normalize_group(test_data, 'prop_country_id', 'comp5_rate_percent_diff'),
'prop_country_id_comp6_rate': normalize_group(test_data, 'prop_country_id', 'comp6_rate'),
'prop_country_id_comp6_inv': normalize_group(test_data, 'prop_country_id', 'comp6_inv'),
'prop_country_id_comp6_rate_percent_diff': normalize_group(test_data, 'prop_country_id', 'comp6_rate_percent_diff'),
'prop_country_id_comp7_rate': normalize_group(test_data, 'prop_country_id', 'comp7_rate'),
'prop_country_id_comp7_inv': normalize_group(test_data, 'prop_country_id', 'comp7_inv'),
'prop_country_id_comp7_rate_percent_diff': normalize_group(test_data, 'prop_country_id', 'comp7_rate_percent_diff'),
'prop_country_id_comp8_rate': normalize_group(test_data, 'prop_country_id', 'comp8_rate'),
'prop_country_id_comp8_inv': normalize_group(test_data, 'prop_country_id', 'comp8_inv'),
'prop_country_id_comp8_rate_percent_diff': normalize_group(test_data, 'prop_country_id', 'comp8_rate_percent_diff'),
'prop_country_id_prop_starrating': normalize_group(test_data, 'prop_country_id', 'prop_starrating'),
'prop_country_id_prop_review_score': normalize_group(test_data, 'prop_country_id', 'prop_review_score'),
'prop_country_id_prop_location_score1': normalize_group(test_data, 'prop_country_id', 'prop_location_score1'),
'prop_country_id_prop_location_score2': normalize_group(test_data, 'prop_country_id', 'prop_location_score2'),
'prop_country_id_prop_log_historical_price': normalize_group(test_data, 'prop_country_id', 'prop_log_historical_price'),
'prop_country_id_price_usd': normalize_group(test_data, 'prop_country_id', 'price_usd'),

})

In [18]:
train_data = pd.concat([train_data, new_train_cols], axis=1)
test_data = pd.concat([test_data, new_test_cols], axis=1)

In [43]:
train_data.fillna(0, inplace=True)
test_data.fillna(0, inplace=True)

In [ ]:
train_data.to_csv("my_train_data.csv", index=False)

In [ ]:
test_data.to_csv("my_test_data.csv", index=False)

## Model

In [44]:
train_data.sort_values("srch_id", inplace=True)

In [45]:
unique_groups = train_data['srch_id'].unique()
shuffled_groups = shuffle(unique_groups, random_state=42)

split_index = int(1 * len(shuffled_groups))
train_groups = shuffled_groups[:split_index]
val_groups = shuffled_groups[split_index:]
train_df = train_data[train_data['srch_id'].isin(train_groups)]
val_df = train_data[train_data['srch_id'].isin(val_groups)]

In [46]:
X_train = train_df.drop(['click_bool', 'booking_bool', 'gross_bookings_usd', 'srch_id', 'position'], axis=1)
y_train = train_df['booking_bool']
X_val = val_df.drop(['click_bool', 'booking_bool', 'gross_bookings_usd', 'srch_id', 'position'], axis=1)
y_val = val_df['booking_bool']

In [47]:
groups_train = train_df.groupby('srch_id').size().values
groups_val = val_df.groupby('srch_id').size().values

In [48]:
train_data = lgb.Dataset(X_train, label=y_train, group=groups_train)
val_data = lgb.Dataset(X_val, label=y_val, group=groups_val)

In [49]:
print(len(X_train))
print(len(X_val))

4958347
0


In [50]:
best_score = 0
best_params = {}

fixed_params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'verbose': -1,
    'ndcg_eval_at': [1, 3, 5, 10],
}

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63, 127],
    'max_depth': [3, 5, 7],
    'min_data_in_leaf': [20, 50, 100],
    'feature_fraction': [0.8, 0.9, 1.0]
}

In [51]:
def perform_cv(params, data, num_boost_round, nfold, seed=42):
    cv_results = lgb.cv(params, data, num_boost_round=num_boost_round, nfold=nfold,
                        seed=seed, stratified=False, callbacks=[lgb.early_stopping(stopping_rounds=10)])
    # Assuming 'ndcg@1-mean' is the metric of interest
    print(f"CV Result Keys: {list(cv_results.keys())}")
    return np.max(cv_results['valid ndcg@1-mean']), cv_results

In [ ]:
for gparams in ParameterGrid(param_grid):
    params = {**fixed_params, **gparams}
    score, results = perform_cv(params, train_data, num_boost_round=100, nfold=5)
    if score > best_score:
        best_score = score
        best_params = gparams
        print(f"New best score: {best_score}")
        print(f"With parameters: {best_params}\n")

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[17]	cv_agg's valid ndcg@1: 0.461413 + 0.00175034	cv_agg's valid ndcg@3: 0.558688 + 0.00146618	cv_agg's valid ndcg@5: 0.600137 + 0.00161589	cv_agg's valid ndcg@10: 0.643619 + 0.00131863
CV Result Keys: ['valid ndcg@1-mean', 'valid ndcg@1-stdv', 'valid ndcg@3-mean', 'valid ndcg@3-stdv', 'valid ndcg@5-mean', 'valid ndcg@5-stdv', 'valid ndcg@10-mean', 'valid ndcg@10-stdv']
New best score: 0.46141294827197876
With parameters: {'feature_fraction': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'min_data_in_leaf': 20, 'num_leaves': 31}

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[17]	cv_agg's valid ndcg@1: 0.461413 + 0.00175034	cv_agg's valid ndcg@3: 0.558688 + 0.00146618	cv_agg's valid ndcg@5: 0.600137 + 0.00161589	cv_agg's valid ndcg@10: 0.643619 + 0.00131863
CV Result Keys: ['valid ndcg@1-mean', 'valid ndcg@1-stdv', 'valid ndcg@3-mean', 'valid n

In [29]:
best_params = {'feature_fraction': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'min_data_in_leaf': 20, 'num_leaves': 31}
full_train_data = lgb.Dataset(X_train, label=y_train, group=groups_train)
params = {**fixed_params, **best_params}
final_model = lgb.train(params, full_train_data, num_boost_round=100)

In [30]:
mod_test_data = test_data.drop(['srch_id'], axis=1)

In [31]:
test_pred = final_model.predict(mod_test_data)
test_data['pred'] = test_pred
sorted_results = test_data.sort_values(by=['srch_id', 'pred'], ascending=[True, False])
submit_data = sorted_results[['srch_id', 'prop_id']]
submit_data.to_csv('4444.csv', index=False)